In [1]:
using Pkg
Pkg.activate("C:/Users/selha/Desktop/MAAAI/environment")  # <-- CAMBIAR A TU RUTA DE ENV
#Pkg.activate("environment")
#Pkg.instantiate()


  Activating project at `C:\Users\selha\Desktop\MAAAI\environment`


In [3]:
Pkg.instantiate()   # solo necesario la primera vez 

In [4]:
using CSV, DataFrames, Statistics, Random
using MLJ
using MLJModelInterface
import MLJBase: transform
const MMI = MLJModelInterface
using HypothesisTests
using DataFramesMeta
using StatsBase
using GLM, StatsModels
using Plots
using Glob
using Flux
using MLJScikitLearnInterface
using MultivariateStats

# PREPARACIÓN DE DATOS
## 1. Carga y unificación de datos

Los datos originales están distribuidos en múltiples ficheros CSV, uno por cada sujeto.  
El objetivo de este bloque es:

- Buscar todos los CSV dentro del directorio raíz.
- Leerlos en memoria de forma homogénea.
- Concatenarlos en un único DataFrame consolidado.
- Guardar `dataset_consolidado.csv` para usarlo en el resto de la práctica.


In [5]:
DATA_ROOT = "C:\\Users\\selha\\Desktop\\MAAAI\\datasets"

function find_all_csv_files(root_dir)
    csv_files = String[]
    for (root, dirs, files) in walkdir(root_dir)
        for file in files
            if endswith(file, ".csv")
                push!(csv_files, joinpath(root, file))
            end
        end
    end
    return csv_files
end

csv_files = find_all_csv_files(DATA_ROOT)
println("Archivos encontrados: ", length(csv_files))

# Leer todos los CSV
dfs = DataFrame[]
for file in csv_files
    try
        df_tmp = CSV.read(file, DataFrame; normalizenames=true)
        push!(dfs, df_tmp)
        println("✓ Leído: ", basename(file), " (", nrow(df_tmp), " filas)")
    catch e
        @warn "No se pudo leer: $file" exception=(e, catch_backtrace())
    end
end

# Consolidar
df_all = vcat(dfs...)
println("\nDataset consolidado: ", nrow(df_all), " filas, ", ncol(df_all), " columnas")

# Guardar
mkpath("data_processed")
CSV.write("data_processed/dataset_consolidado.csv", df_all)
println(" Guardado: data_processed/dataset_consolidado.csv")

Archivos encontrados: 30
✓ Leído: Sujeto_01.csv (347 filas)
✓ Leído: Sujeto_05.csv (302 filas)
✓ Leído: Sujeto_07.csv (308 filas)
✓ Leído: Sujeto_11.csv (316 filas)
✓ Leído: Sujeto_03.csv (341 filas)
✓ Leído: Sujeto_09.csv (288 filas)
✓ Leído: Sujeto_23.csv (372 filas)
✓ Leído: Sujeto_25.csv (409 filas)
✓ Leído: Sujeto_15.csv (328 filas)
✓ Leído: Sujeto_17.csv (368 filas)
✓ Leído: Sujeto_21.csv (408 filas)
✓ Leído: Sujeto_13.csv (327 filas)
✓ Leído: Sujeto_19.csv (360 filas)
✓ Leído: Sujeto_27.csv (376 filas)
✓ Leído: Sujeto_29.csv (344 filas)
✓ Leído: Sujeto_02.csv (302 filas)
✓ Leído: Sujeto_04.csv (317 filas)
✓ Leído: Sujeto_06.csv (325 filas)
✓ Leído: Sujeto_08.csv (281 filas)
✓ Leído: Sujeto_10.csv (294 filas)
✓ Leído: Sujeto_12.csv (320 filas)
✓ Leído: Sujeto_14.csv (323 filas)
✓ Leído: Sujeto_16.csv (366 filas)
✓ Leído: Sujeto_18.csv (364 filas)
✓ Leído: Sujeto_20.csv (354 filas)
✓ Leído: Sujeto_22.csv (321 filas)
✓ Leído: Sujeto_24.csv (381 filas)
✓ Leído: Sujeto_26.csv (392 fi

## 2. Resumen del conjunto de datos

En esta sección obtenemos una descripción básica del dataset consolidado:

- Número total de instancias (filas)
- Número total de variables
- Número de individuos (`subject`)
- Número de clases de salida (`Activity`)


In [6]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

num_variables_totales = ncol(df)
num_instancias = nrow(df)

# Número de individuos
if "subject" in names(df)
    num_individuos = length(unique(df.subject))
else
    @warn "No se encontró la columna 'subject'; no se puede calcular el número de individuos."
    num_individuos = missing
end

# Número de clases de salida
if "Activity" in names(df)
    num_clases_salida = length(unique(df.Activity))
else
    @warn "No se encontró la columna 'Activity'; no se puede calcular el número de clases de salida."
    num_clases_salida = missing
end

# Variables de entrada (todas excepto subject + Activity)
num_features = num_variables_totales - 2

println("\n=== Resumen del dataset ===")
println("Número total de variables (incluyendo subject y Activity): ", num_variables_totales)
println("Número de variables de características: ", num_features)
println("Número de instancias: ", num_instancias)
println("Número de individuos: ", num_individuos)
println("Número de clases de salida: ", num_clases_salida)
println("=============================================")


=== Resumen del dataset ===
Número total de variables (incluyendo subject y Activity): 563
Número de variables de características: 561
Número de instancias: 10299
Número de individuos: 30
Número de clases de salida: 6


## 3. Análisis de valores ausentes

En esta sección calculamos:

- El **porcentaje de valores nulos por variable**  
- El **porcentaje total de valores nulos** en el dataset  

Esto permite entender la magnitud del problema de valores faltantes y justificar
posteriormente el método de imputación empleado (subject-wise con media/mediana).

In [7]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

# Análisis de valores ausentes
n_rows = nrow(df)
n_cols = ncol(df)

# Porcentaje de nulos por columna
porc_nulos_col = Dict{String, Float64}()

for col in names(df)
    n_missing = count(ismissing, df[!, col])
    porc_nulos_col[col] = 100 * n_missing / n_rows
end

# Porcentaje total de nulos en todo el dataset
total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
total_values = n_rows * n_cols
porc_total_missing = 100 * total_missing / total_values

println("=== Porcentaje de valores nulos por columna ===")
for (col, pct) in sort(collect(porc_nulos_col); by = x -> x[2], rev = true)
    println(rpad(col, 30), ": ", round(pct, digits = 2), "%")
end

println("\nPorcentaje total de valores nulos en el dataset: ",
        round(porc_total_missing, digits = 2), "%")
println("===============================================")

=== Porcentaje de valores nulos por columna ===
tBodyGyroMag_mad_             : 10.03%
tBodyGyroMag_iqr_             : 10.03%
fBodyAcc_mad_Y                : 10.02%
fBodyAccJerk_mean_X           : 10.02%
fBodyBodyGyroMag_iqr_         : 10.0%
fBodyAcc_std_X                : 10.0%
tBodyAccMag_max_              : 10.0%
tGravityAccMag_std_           : 10.0%
tGravityAccMag_entropy_       : 10.0%
tBodyAccJerk_entropy_Y        : 10.0%
tBodyAccJerk_energy_X         : 10.0%
tBodyGyro_arCoeff_Y_2         : 9.99%
tBodyGyroJerk_arCoeff_Z_2     : 9.99%
fBodyAcc_maxInds_Y            : 9.99%
fBodyBodyGyroJerkMag_energy_  : 9.99%
fBodyGyro_mean_X              : 9.99%
fBodyGyro_maxInds_Z           : 9.99%
fBodyAccJerk_bandsEnergy_49_56_2: 9.99%
tBodyAcc_entropy_Z            : 9.99%
fBodyGyro_energy_Y            : 9.99%
tGravityAcc_std_Y             : 9.99%
fBodyAcc_kurtosis_Y           : 9.99%
tBodyAccJerkMag_arCoeff_2     : 9.99%
fBodyGyro_bandsEnergy_49_56_1 : 9.99%
tGravityAcc_max_X             : 9.

## 4. Imputación de valores ausentes (subject-wise)

En esta sección imputamos los valores faltantes de las variables numéricas siguiendo
un criterio **por sujeto**:

- Para cada sujeto (`subject`) se toman únicamente sus propias observaciones.
- Para cada columna numérica con valores ausentes:
  - Se comprueba si hay outliers mediante el rango intercuartílico (IQR).
  - Si **hay outliers**, se imputa con la **mediana**.
  - Si **no hay outliers**, se imputa con la **media**.
- No se modifican las columnas `subject` ni `Activity`.

El resultado es un nuevo dataset imputado que conserva la estructura original pero sin
valores faltantes en las variables numéricas.


In [8]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)


# -----------------------------------------------------------
# Función auxiliar para detectar columnas numéricas (permitiendo Missing)
# -----------------------------------------------------------
function col_contains_numeric(eltyp)
    if eltyp <: Real
        return true
    end
    try
        for t in Base.uniontypes(eltyp)
            if t <: Real
                return true
            end
        end
    catch
    end
    return false
end

# -----------------------------------------------------------
# Detección de outliers con IQR
# -----------------------------------------------------------
function tiene_outliers(vals)
    q1 = quantile(vals, 0.25)
    q3 = quantile(vals, 0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    any(x -> x < lower || x > upper, vals)
end

# -----------------------------------------------------------
# Imputación subject-wise
# -----------------------------------------------------------
function impute_subjectwise(df::DataFrame)
    df_imp = deepcopy(df)

    excluded = Set(["subject", "Activity"])   # no se imputan estas columnas

    # Detectar columnas numéricas a imputar
    numeric_cols = String[]
    for c in names(df_imp)
        if c ∉ excluded && col_contains_numeric(eltype(df_imp[!, c]))
            push!(numeric_cols, c)
        end
    end

    subjects = unique(df_imp.subject)

    println("Columnas numéricas a imputar: ", length(numeric_cols))
    println("Sujetos encontrados: ", length(subjects))

    for s in subjects
        rows_subject = df_imp.subject .== s

        for col in numeric_cols
            colvec = df_imp[rows_subject, col]
            nmiss = count(ismissing, colvec)
            if nmiss == 0
                continue
            end

            nonmiss = collect(skipmissing(colvec))
            if isempty(nonmiss)
                continue
            end

            method = tiene_outliers(nonmiss) ? "median" : "mean"
            value  = method == "median" ? median(nonmiss) : mean(nonmiss)

            mask = rows_subject .& ismissing.(df_imp[!, col])
            df_imp[mask, col] .= value
        end
    end

    return df_imp
end


# -----------------------------------------------------------
# Aplicar imputación y guardar resultado
# -----------------------------------------------------------
df_imputed = impute_subjectwise(df)

println("\nDataset imputado: ", nrow(df_imputed), " filas, ", ncol(df_imputed), " columnas.")

CSV.write("data_processed/dataset_consolidado_imputed.csv", df_imputed)

println("Guardado en: data_processed/dataset_consolidado_imputed.csv")


Columnas numéricas a imputar: 561
Sujetos encontrados: 30

Dataset imputado: 10299 filas, 563 columnas.
Guardado en: data_processed/dataset_consolidado_imputed.csv


## 5. Partición holdout (10 % de sujetos)

En este bloque se reserva un **10 % de los sujetos completos** como conjunto de
**test final (holdout)**, siguiendo las indicaciones del enunciado:

- Se parte del dataset ya imputado (`df_imputed`).
- Se obtienen todos los identificadores de sujetos (`subject`).
- Con la semilla `104` se selecciona aleatoriamente el 10 % de los sujetos.
- Todas las filas de esos sujetos pasan a formar el conjunto **test**.
- El resto de sujetos componen el conjunto **train**.

Este conjunto de test **no se utiliza en la validación cruzada** y se reserva
exclusivamente para la evaluación final de los modelos seleccionados.

In [9]:
# Cargar dataset imputado desde data_processed
df_imputed = CSV.read("data_processed/dataset_consolidado_imputed.csv", DataFrame)

println("Sujetos detectados en el dataset imputado:")
subjects = unique(df_imputed.subject)
println(subjects)

# Semilla pedida en el enunciado
Random.seed!(104)

# 10% de sujetos → al menos 1
n_test = max(1, round(Int, length(subjects) * 0.10))

println("Número total de sujetos: ", length(subjects))
println("Número de sujetos para TEST (10%): ", n_test)

# Selección aleatoria reproducible
test_subjects = Random.shuffle(subjects)[1:n_test]

println("\n=== Sujetos seleccionados para TEST (holdout) ===")
println(test_subjects)

# Máscaras de pertenencia
test_mask  = in.(df_imputed.subject, Ref(test_subjects))
train_mask = .!test_mask

# Particionar
df_test  = df_imputed[test_mask, :]
df_train = df_imputed[train_mask, :]

println("\nFilas train: ", nrow(df_train))
println("Filas test : ", nrow(df_test))

# ------------------ Guardado en data_processed ------------------

CSV.write("data_processed/dataset_train.csv", df_train)
CSV.write("data_processed/dataset_test.csv", df_test)

# Lista de sujetos del test
ts_df = DataFrame(subject = test_subjects)
CSV.write("data_processed/test_subjects.csv", ts_df)

# Informe del número de muestras por sujeto en el dataset completo
counts = combine(groupby(df_imputed, :subject), nrow => :n_rows)
CSV.write("data_processed/subject_counts.csv", counts)

println("\nArchivos guardados en data_processed/:")
println(" - Train dataset:        dataset_train.csv")
println(" - Test dataset:         dataset_test.csv")
println(" - Test subjects list:   test_subjects.csv")
println(" - Subject counts:       subject_counts.csv")


Sujetos detectados en el dataset imputado:
[1, 5, 7, 11, 3, 9, 23, 25, 15, 17, 21, 13, 19, 27, 29, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
Número total de sujetos: 30
Número de sujetos para TEST (10%): 3

=== Sujetos seleccionados para TEST (holdout) ===
[25, 18, 22]

Filas train: 9205
Filas test : 1094

Archivos guardados en data_processed/:
 - Train dataset:        dataset_train.csv
 - Test dataset:         dataset_test.csv
 - Test subjects list:   test_subjects.csv
 - Subject counts:       subject_counts.csv


## 6. Validación cruzada individual-wise (5-Fold)

Tras aplicar la partición *holdout*, usamos únicamente el conjunto de entrenamiento
(`df_train`) para construir una validación cruzada 5-fold basada en **sujetos**:

- Cada fold contiene un subconjunto de sujetos completos.
- En cada fold, uno (o varios) sujetos se usan como **test interno**.
- El resto se usan como **train**.
- Nunca se mezclan instancias de un mismo sujeto entre train y test.
- Se fija la semilla `104` para reproducibilidad.

Este esquema es imprescindible porque los datos están fuertemente correlacionados por
sujeto; por tanto, una validación aleatoria estándar produciría *data leakage*.

In [10]:
# Cargar el conjunto de entrenamiento generado en el holdout
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)

# Sujetos disponibles en TRAIN
subjects_train = unique(df_train.subject)
n_subjects = length(subjects_train)
n_folds = 5

println("Sujetos disponibles en TRAIN: ", n_subjects)

# -----------------------------------------------------------
# Función generadora de folds balanceados
# -----------------------------------------------------------
function generate_subjectwise_folds(subjects::Vector, k::Int=5; seed=104)
    Random.seed!(seed)
    shuffled = Random.shuffle(subjects)

    base_size = div(length(shuffled), k)
    extra = mod(length(shuffled), k)

    folds = Vector{Vector{eltype(subjects)}}()
    start_idx = 1

    for i in 1:k
        fold_size = base_size + (i <= extra ? 1 : 0)
        push!(folds, shuffled[start_idx:start_idx+fold_size-1])
        start_idx += fold_size
    end

    return folds
end

folds = generate_subjectwise_folds(subjects_train, n_folds)

println("\nSujetos por fold:")
for i in 1:length(folds)
    println("Fold $i: ", folds[i])
end

# -----------------------------------------------------------
# Crear los CSV de train/test por fold
# -----------------------------------------------------------

# Crear la carpeta de salida si no existe
mkpath("data_processed/folds")

for i in 1:n_folds
    fold_subjects = folds[i]

    # Elegimos 1 sujeto como test interno (igual que tu script original)
    fold_subjects_shuffled = Random.shuffle(copy(fold_subjects))
    test_subject = fold_subjects_shuffled[1]        # sujeto de test interno
    train_subjects = fold_subjects_shuffled[2:end]  # resto son train

    println("\nFold $i")
    println("  Sujeto de test interno: ", test_subject)
    println("  Sujetos de train: ", train_subjects)

    test_mask  = in.(df_train.subject, Ref([test_subject]))
    train_mask = in.(df_train.subject, Ref(train_subjects))

    df_fold_train = df_train[train_mask, :]
    df_fold_test  = df_train[test_mask, :]

    # Guardar CSVs en data_processed/folds/
    CSV.write("data_processed/folds/fold$(i)_train.csv", df_fold_train)
    CSV.write("data_processed/folds/fold$(i)_test.csv",  df_fold_test)
end

println("\nValidación cruzada individual-wise 5-fold generada correctamente.")
println("Archivos guardados en data_processed/folds/")


Sujetos disponibles en TRAIN: 27

Sujetos por fold:
Fold 1: [15, 24, 28, 26, 29, 16]
Fold 2: [27, 2, 7, 9, 23, 21]
Fold 3: [10, 6, 30, 12, 4]
Fold 4: [3, 19, 17, 11, 1]
Fold 5: [8, 14, 5, 20, 13]

Fold 1
  Sujeto de test interno: 26
  Sujetos de train: [28, 24, 16, 15, 29]

Fold 2
  Sujeto de test interno: 2
  Sujetos de train: [9, 27, 23, 7, 21]

Fold 3
  Sujeto de test interno: 4
  Sujetos de train: [12, 30, 6, 10]

Fold 4
  Sujeto de test interno: 3
  Sujetos de train: [11, 19, 1, 17]

Fold 5
  Sujeto de test interno: 5
  Sujetos de train: [8, 14, 20, 13]

Validación cruzada individual-wise 5-fold generada correctamente.
Archivos guardados en data_processed/folds/


### 7. Normalización Min-Max con un nodo MLJ personalizado

La rúbrica de la práctica exige que la normalización se implemente como un **Nodo de MLJ**, y no como
una operación manual. El objetivo es garantizar que:

- El normalizador se ajusta **únicamente con el conjunto de entrenamiento**.
- El mismo transformador se aplica después sobre validaciones internas y sobre el conjunto de test.
- Se evita cualquier **fuga de información**.
- La normalización pueda integrarse dentro de un **pipeline de MLJ** o del proceso de validación cruzada.

En nuestro entorno concreto, los transformadores predefinidos de MLJ
(`Standardizer`, `FeatureRescaler`, `UnivariateStandardizer`, etc.) no estaban disponibles en la
versión de `MLJModels` instalada.  
Para seguir estrictamente la rúbrica, optamos por implementar un **nodo MLJ propio**, totalmente
compatible con la interfaz de MLJ.

Este nodo (`MyMinMaxScaler`) implementa:

- `fit(model, X)` → calcula los mínimos y máximos por columna únicamente a partir del conjunto *train*  
- `transform(model, X)` → aplica la fórmula del Min-Max scaling  
- Exclusión automática de columnas no numéricas  
- Ignora explícitamente las columnas `subject` y `Activity`, que no deben normalizarse  
- Es robusto ante valores `missing`  

Con este nodo se obtiene un funcionamiento equivalente al Min-Max tradicional,  
pero **respetando la filosofía y requisitos formales de MLJ**.

In [11]:
using MLJ
using MLJBase
#-----------------------------------------------------------
# Definición del escalador Min-Max personalizado
#-----------------------------------------------------------
const MMI = MLJModelInterface

struct MyMinMaxScaler <: MMI.Unsupervised
    ignore::Vector{Symbol}
end

MyMinMaxScaler(; ignore = [:subject, :Activity]) = MyMinMaxScaler(ignore)

#-----------------------------------------------------------
# Implementación de fit y transform
#-----------------------------------------------------------
function MMI.fit(model::MyMinMaxScaler, verbosity::Int, X)

    # 1. columnas realmente numéricas
    numeric_cols = [
        c for c in names(X)
        if !(c in model.ignore) &&
           all(x -> x === missing || x isa Real, X[!, c])
    ]

    mins = Dict{String, Float64}()
    maxs = Dict{String, Float64}()

    for col in numeric_cols
        col_data = collect(skipmissing(X[!, col]))

        if isempty(col_data)
            mins[col] = 0.0
            maxs[col] = 0.0
        else
            mins[col] = minimum(col_data)
            maxs[col] = maximum(col_data)
        end
    end

    fitresult = (
        mins = mins,
        maxs = maxs,
        numeric_cols = numeric_cols
    )

    return fitresult, nothing, nothing
end


function MMI.transform(model::MyMinMaxScaler, fitresult, X)
    X_new = deepcopy(X)

    for col in fitresult.numeric_cols
        minv = fitresult.mins[col]
        maxv = fitresult.maxs[col]

        if maxv != minv
            X_new[!, col] = (X_new[!, col] .- minv) ./ (maxv - minv)
        else
            X_new[!, col] .= 0.0
        end
    end

    return X_new
end

#-----------------------------------------------------------
# Aplicar el escalado Min-Max y guardar los datasets escalados
#-----------------------------------------------------------

# Cargar los datasets de train y test desde data_processed
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)
df_test  = CSV.read("data_processed/dataset_test.csv", DataFrame)

scaler = MyMinMaxScaler(ignore = [:subject, :Activity])

mach = machine(scaler, df_train)
MLJBase.fit!(mach)

df_train_scaled = MLJBase.transform(mach, df_train)
df_test_scaled  = MLJBase.transform(mach, df_test)

CSV.write("data_processed/dataset_train_scaled.csv", df_train_scaled)
CSV.write("data_processed/dataset_test_scaled.csv", df_test_scaled)

[ Info: Training machine(MyMinMaxScaler(ignore = [:subject, :Activity]), …).


"data_processed/dataset_test_scaled.csv"

# MODELOS BÁSICOS Y SELECCIÓN DE ATRIBUTOS

### Selector sin reducción (baseline)

Este selector se usa como referencia: no elimina ninguna característica,
pero permite integrarlo como nodo MLJ en pipelines y compararlo con las demás técnicas
de selección de características.

In [12]:
using MLJModelInterface
const MMI = MLJModelInterface

# =====================================
# Selector de características: Sin reducción
# =====================================

"""
    SelectorNone()

Selector que no elimina ninguna característica.
Devuelve todas las columnas excepto :subject y :Activity.
"""
struct SelectorNone <: MMI.Unsupervised end
function MMI.fit(model::SelectorNone, verbosity::Int, X)
    # FIX: Usar Strings para filtrar ("subject", "Activity")
    all_cols = names(X)
    exclude = ["subject", "Activity"]
    selected = filter(c -> c ∉ exclude, all_cols)
    return (selected=selected,), nothing, nothing
end
function MMI.transform(model::SelectorNone, fitresult, X)
    return select(X, fitresult.selected)
end

### Selección de características ANOVA (F-test)

Para cada característica numérica se calcula un test ANOVA univariante tomando como
variable dependiente la clase `Activity`. Se seleccionan las 50 características con
mayor estadístico F. Este filtro se ajusta únicamente sobre los datos de entrenamiento
para evitar fuga de información.


In [13]:

# ===============================
# Definición del modelo 
# ===============================

"""
    SelectorANOVA(k = 50)

Selector de características basado en One-Way ANOVA.
Selecciona las k columnas con mayor estadístico F.
"""
struct SelectorANOVA <: MMI.Unsupervised
    k::Int
end

SelectorANOVA(; k = 50) = SelectorANOVA(k)


# ===============================
# Fit (CORREGIDO)
# ===============================

function MMI.fit(model::SelectorANOVA, verbosity::Int, X)

    # 1) Asegurar target correcto (CORREGIDO: String en lugar de Symbol)
    @assert "Activity" ∈ names(X) "El dataset debe contener la columna 'Activity' como target. Columnas presentes: $(names(X))"

    # Acceso seguro a la columna Activity (ya sea String o Symbol en el DataFrame)
    y = X.Activity

    # Convertir Activity a categórica si es necesario (maneja String31)
    if !(y isa CategoricalVector)
        y = categorical(String.(y)) # Forzamos conversión a String estándar
    end

    classes = levels(y)

    # 2) Seleccionar columnas de features (CORREGIDO: Strings)
    feature_cols = String[]
    all_names = names(X)
    
    for col in all_names
        # Filtramos explícitamente las columnas que no son features
        if col != "subject" && col != "Activity"
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = String[]

    # 3) Evaluar ANOVA por feature
    for col in feature_cols
        # Accedemos usando String
        coldata = X[!, col]

        # Imputación de missing: media
        if any(ismissing, coldata)
            non_missing = skipmissing(coldata)
            if isempty(non_missing)
                push!(scores, -Inf)
                push!(cols, col)
                continue
            end
            μ = mean(non_missing)
            coldata = coalesce.(coldata, μ)
        end

        # Feature constante → no aporta
        if length(unique(coldata)) < 2
            push!(scores, -Inf)
            push!(cols, col)
            continue
        end

        # Construcción de grupos por clase
        groups = Vector{Vector{Float64}}()
        valid_feature = true

        for cls in classes
            # Filtrado booleano seguro
            vals = coldata[y .== cls]

            # ANOVA necesita al menos 2 observaciones por grupo
            if length(vals) <= 1
                valid_feature = false
                break
            end

            push!(groups, collect(Float64, vals)) # Asegurar tipo Float64
        end

        if !valid_feature
            push!(scores, -Inf)
            push!(cols, col)
            continue
        end

        # ANOVA
        try
            test = OneWayANOVA(groups...)
            push!(scores, test.F)
        catch
            push!(scores, -Inf)
        end

        push!(cols, col)
    end

    # 4) Ranking y selección final
    order = sortperm(scores, rev = true)
    
    # Asegurar que no pedimos más k de las columnas disponibles
    real_k = min(model.k, length(cols))
    
    if real_k == 0
        selected = String[]
    else
        selected = cols[order][1:real_k]
    end

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end


# ===============================
# Transform (CORREGIDO)
# ===============================

function MMI.transform(model::SelectorANOVA, fitresult, X)
    # Devolvemos un DataFrame solo con las columnas seleccionadas
    # select(X, names) es más seguro que X[:, names] en DataFrames nuevos
    return select(X, fitresult.selected)
end

### Filtrado de características mediante correlación de Pearson

PROBLEMA: Pearson necesita que todo sea numérico, pero nuestra target es categórica (se prueba igual porque el señor lo pide)

In [14]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using CategoricalArrays

# ==============================
# Definición del modelo (Notebook)
# ==============================

"""
    SelectorPearson(k = 50)

Selector de características basado en correlación de Pearson.
Convierte Activity a índices numéricos y calcula |r|
entre cada feature y la variable objetivo.
"""
struct SelectorPearson <: MMI.Unsupervised
    k::Int
end

SelectorPearson(; k = 50) = SelectorPearson(k)


# ==============================
# Fit
# ==============================

function MMI.fit(model::SelectorPearson, verbosity::Int, X)

    # 1) Preparar Activity (CORREGIDO: Busca String "Activity")
    # DataFrames suele devolver nombres como Strings
    if "Activity" ∉ names(X)
        # Fallback por si acaso en alguna versión devuelve symbols
        if :Activity ∉ propertynames(X)
             throw(AssertionError("El dataset debe contener la columna 'Activity' como target."))
        end
        ycat = X.Activity
    else
        ycat = X[!, "Activity"]
    end

    if !(ycat isa CategoricalVector)
        ycat = categorical(ycat)
    end

    # Convertir actividades a índices numéricos (1..C)
    y = levelcode.(ycat)

    # 2) Seleccionar columnas de features (CORREGIDO: Strings)
    feature_cols = String[]
    for col in names(X)
        # Filtramos usando Strings
        if col != "subject" && col != "Activity"
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = String[]

    # 3) Pearson por feature
    for col in feature_cols
        x = X[!, col]

        # Si es constante → no aporta
        if length(unique(skipmissing(x))) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Alinear missing
        mask = .!ismissing.(x)
        
        # Conversión segura a Float64
        x_clean = Float64.(collect(skipmissing(x)))
        
        # Ajustar y al tamaño de x_clean (si hubiera missings)
        # Nota: skipmissing ya filtra, pero necesitamos los índices correspondientes en y
        # Simplificación: Asumimos que si X tiene missing, esa fila se ignora
        y_clean = y[mask]

        # Pocos datos → score 0
        if length(x_clean) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Calcular correlación
        try
            r = cor(x_clean, y_clean)
            push!(scores, abs(r))
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # 4) Ordenar y seleccionar
    order = sortperm(scores, rev=true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end

# ==============================
# Transform
# ==============================

function MMI.transform(model::SelectorPearson, fitresult, X)
    # DataFrames.select acepta vector de Strings
    return select(X, fitresult.selected)
end

### Filtrado Spearman

Spearman mide la correlación por rangos. Es más robusto ante relaciones no lineales.
Se seleccionan las 50 características con mayor |ρ|.


In [15]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using StatsBase
using CategoricalArrays
using DataFrames # Necesario para select

# ==============================
# Definición del modelo (Corregido)
# ==============================

"""
    SelectorSpearman(k = 50)

Selector de características basado en correlación de Spearman.
Convierte Activity en códigos numéricos y calcula |ρ|
entre cada feature y la variable objetivo.
"""
struct SelectorSpearman <: MMI.Unsupervised
    k::Int
end

SelectorSpearman(; k=50) = SelectorSpearman(k)


# ==============================
# Fit (CORREGIDO)
# ==============================

function MMI.fit(model::SelectorSpearman, verbosity::Int, X)

    # 1) Asegurar target correcto (CORREGIDO: String)
    @assert "Activity" ∈ names(X) "El dataset debe contener la columna 'Activity' como target."

    # Convertir Activity a categórica asegurando String estándar
    ycat = X.Activity
    if !(ycat isa CategoricalVector)
        ycat = categorical(String.(ycat))
    end

    # Convertir categorías a códigos numéricos 1..C
    y = levelcode.(ycat)

    # 2) Seleccionar columnas de features (CORREGIDO: String[])
    feature_cols = String[]
    for col in names(X)
        if col != "subject" && col != "Activity"
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = String[]

    # 3) Calcular Spearman por feature
    for col in feature_cols
        # Acceso usando String
        x = X[!, col]

        # Quitar missing alineados
        mask = .!ismissing.(x)
        
        # Si x tiene missings, usamos skipmissing, pero alineamos y con la mascara
        if any(ismissing, x)
             x_clean = Float64.(collect(skipmissing(x)))
             y_clean = y[mask]
        else
             x_clean = Float64.(x)
             y_clean = y
        end

        # Si el feature es constante o insuficiente → score 0
        if length(unique(x_clean)) < 2 || length(x_clean) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Intentar correlación
        try
            # corspearman de StatsBase
            r = corspearman(x_clean, y_clean)
            push!(scores, abs(r))
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # 4) Ordenar y seleccionar top-k
    order = sortperm(scores, rev=true)
    real_k = min(model.k, length(cols))
    
    if real_k == 0
        selected = String[]
    else
        selected = cols[order][1:real_k]
    end

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end


# ==============================
# Transform (CORREGIDO)
# ==============================

function MMI.transform(model::SelectorSpearman, fitresult, X)
    return select(X, fitresult.selected)
end

### Filtrado Kendall Tau

El coeficiente Tau de Kendall es un estimador no paramétrico basado en concordancias
y discordancias entre pares. Resulta más estable con datos con ruido.

Se seleccionan las 50 mejores características por |τ|.


In [16]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using StatsBase
using CategoricalArrays

# ==============================
# Definición del modelo (Notebook)
# ==============================

"""
    SelectorKendall(k = 50)

Selector de características basado en correlación de Kendall Tau.
Convierte Activity a códigos numéricos y calcula |τ|
entre cada feature y la variable objetivo.
"""
struct SelectorKendall <: MMI.Unsupervised
    k::Int
end

SelectorKendall(; k = 50) = SelectorKendall(k)


# ==============================
# Fit (CORREGIDO)
# ==============================

function MMI.fit(model::SelectorKendall, verbosity::Int, X)

    # 1) Comprobar columna Activity (CORREGIDO: String)
    if "Activity" ∉ names(X)
         throw(AssertionError("El dataset debe contener la columna 'Activity'."))
    end

    # Acceso seguro a la columna
    ycat = X[!, "Activity"]

    if !(ycat isa CategoricalVector)
        ycat = categorical(ycat)
    end

    # Convertir categorías a índices numéricos 1..C
    y = levelcode.(ycat)

    # 2) Seleccionar columnas de features (CORREGIDO: Strings)
    feature_cols = String[]
    for col in names(X)
        if col != "subject" && col != "Activity"
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = String[]

    # 3) Calcular Kendall Tau para cada feature
    for col in feature_cols
        x = X[!, col]

        # Alinear missing
        mask = .!ismissing.(x)
        x_clean = Float64.(collect(skipmissing(x))) # collect para asegurar vector
        y_clean = y[mask]

        # Feature constante o con muy pocos datos → score 0
        if length(unique(x_clean)) < 2 || length(x_clean) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Kendall Tau
        try
            # corkendall suele venir de KendallTau.jl o StatsBase recientes
            τ = StatsBase.corkendall(x_clean, y_clean)
            push!(scores, abs(τ))
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # 4) Ordenar y seleccionar las k mejores
    order = sortperm(scores, rev=true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected=selected,)
    report = (scores=scores, selected=selected)

    return fitresult, nothing, report
end


# ==============================
# Transform (CORREGIDO)
# ==============================

function MMI.transform(model::SelectorKendall, fitresult, X)
    # select con vector de Strings es seguro
    return select(X, fitresult.selected)
end

### Filtrado por Información Mutua (MI)

La información mutua permite capturar dependencias no lineales entre cada característica
y la variable objetivo. Dado que la versión de MLJBase instalada no incluye la función
`mutualinfo`, se ha implementado una versión personalizada basada en histogramas, que 
calcula MI de forma robusta sin dependencias externas.

De cada característica se obtiene su MI con la clase, y se seleccionan las 50 con mayor valor.

In [17]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using StatsBase # Necesario para Histogram, countmap, binindex
using CategoricalArrays
using DataFrames

# ==============================
# Función Auxiliar: Mutual Information
# ==============================

"""
    mutual_information(x, y; bins=10)

Calcula la información mutua entre dos vectores x e y.
Si x o y son continuos, se discretizan automáticamente en `bins` estratos.
Se ignoran valores missing de manera alineada.
"""
function mutual_information(x, y; bins=10)

    # 1. Eliminar missing alineados
    mask = .!(ismissing.(x) .| ismissing.(y))
    x_clean = x[mask]
    y_clean = y[mask]
    
    if isempty(x_clean) return 0.0 end

    # 2. Discretizar si son continuos (Float)
    # Convertimos a vector concreto para evitar errores con tipos Union
    if eltype(x_clean) <: AbstractFloat
        # fit(Histogram) puede fallar si todos los valores son iguales
        if length(unique(x_clean)) < 2
            x_disc = ones(Int, length(x_clean))
        else
            hx = fit(Histogram, x_clean, nbins=bins)
            x_disc = StatsBase.binindex.(Ref(hx), x_clean)
        end
    else
        x_disc = x_clean
    end

    if eltype(y_clean) <: AbstractFloat
        if length(unique(y_clean)) < 2
            y_disc = ones(Int, length(y_clean))
        else
            hy = fit(Histogram, y_clean, nbins=bins)
            y_disc = StatsBase.binindex.(Ref(hy), y_clean)
        end
    else
        y_disc = y_clean
    end

    # 3. Contar probabilidades
    n = length(x_disc)
    px  = countmap(x_disc)
    py  = countmap(y_disc)
    pxy = countmap(zip(x_disc, y_disc))

    # 4. Calcular MI
    mi = 0.0
    for ((xi, yi), nxy) in pxy
        px_i = px[xi]
        py_i = py[yi]

        pxy_p = nxy / n
        px_p  = px_i / n
        py_p  = py_i / n

        # Usamos eps() para evitar log(0)
        mi += pxy_p * log(pxy_p / (px_p * py_p + eps()))
    end

    return mi
end


# ==============================
# Definición del modelo
# ==============================

"""
SelectorMI(k=50)

Selector de características basado en Información Mutua.
Selecciona los k features con mayor MI respecto a Activity.
"""
struct SelectorMI <: MMI.Unsupervised
    k::Int
end

SelectorMI(; k=50) = SelectorMI(k)

# ==============================
# Fit (CORREGIDO)
# ==============================

function MMI.fit(model::SelectorMI, verbosity::Int, X)

    # CORRECCIÓN 1: Usar String en lugar de Symbol
    @assert "Activity" ∈ names(X) "El dataset debe contener la columna 'Activity'."

    # Convertir Activity a vector numérico
    # CORRECCIÓN 2: Asegurar conversión a String estándar antes de categorical
    y_raw = X.Activity
    if !(y_raw isa CategoricalVector)
        ycat = categorical(String.(y_raw))
    else
        ycat = y_raw
    end
    y = levelcode.(ycat)

    # Identificar columnas de features
    # CORRECCIÓN 3: Usar String[] y comparar con Strings
    feature_cols = String[]
    for col in names(X)
        if col != "subject" && col != "Activity"
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = String[]

    # Calcular MI para cada feature
    for col in feature_cols
        # Acceso por String
        x = X[!, col]

        try
            # Convertir a Float64 si es posible para la función auxiliar
            if eltype(x) <: Number
                 x_val = Float64.(x)
            else
                 x_val = x
            end
            
            mi = mutual_information(x_val, y)
            push!(scores, mi)
        catch e
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # Selección top-k
    order = sortperm(scores, rev=true)
    real_k = min(model.k, length(cols))
    
    if real_k == 0
        selected = String[]
    else
        selected = cols[order][1:real_k]
    end

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end

# ==============================
# Transform (CORREGIDO)
# ==============================

function MMI.transform(model::SelectorMI, fitresult, X)
    # CORRECCIÓN 4: Usar select para robustez
    return select(X, fitresult.selected)
end

### Filtrado RFE (Recursive Feature Elimination) con Regresión Logística

El enunciado especifica que el método RFE debe utilizar una regresión logística,
eliminando el 50 % de las características en cada iteración. 

Dado que la versión de MLJModels disponible en este entorno no incluye un modelo
de regresión logística, se ha implementado el RFE mediante `GLM.jl`, el paquete
estándar de Julia para modelos lineales generalizados.

El procedimiento es el siguiente:

1. Se toma el conjunto completo de características numéricas.
2. Se ajusta una regresión logística (`glm`) con todas ellas.
3. Se ordenan las características según la magnitud absoluta de sus coeficientes.
4. Se elimina el 50 % menos relevante.
5. El proceso se repite hasta conservar exactamente **50 características**.

Este nodo sigue estrictamente la rúbrica y el enunciado, 
y se integra en MLJ mediante la interfaz `fit` → `transform`.


In [18]:
using MLJModelInterface
using GLM, StatsModels
using CategoricalArrays
using Statistics
using DataFrames

const MMI = MLJModelInterface

# ========================================================
# Definición del modelo
# ========================================================

struct SelectorRFE <: MMI.Unsupervised
    k::Int     # número final de variables
end

SelectorRFE(; k=50) = SelectorRFE(k)


# ========================================================
# Función auxiliar: importancia multiclase One-vs-Rest
# ========================================================

"""
    feature_importance_multiclass(X, y, features)

Entrena un modelo GLM con INTERCEPTO para cada clase (one-vs-rest) 
y devuelve la importancia global de cada feature.
"""
function feature_importance_multiclass(X, y, features)

    # 1. Conversión robusta a Matrix{Float64}
    # Usamos acceso por String (f es String)
    cols_data = [Vector{Float64}(X[!, f]) for f in features]
    X_pure = reduce(hcat, cols_data)
    
    # 2. Añadir columna de Intercepto
    # GLM.jl matrix interface no añade intercepto automáticamente en este modo.
    n_rows = size(X_pure, 1)
    X_glm = hcat(ones(n_rows), X_pure) 

    y_cat = categorical(y)
    classes = levels(y_cat)

    n_features = length(features)
    importances = zeros(Float64, n_features)
    
    successful_classes = 0

    # Para cada clase: modelo binario (clase vs resto)
    for cls in classes
        # Target binario: 1.0 si es la clase, 0.0 si no
        y_bin = ifelse.(y_cat .== cls, 1.0, 0.0)

        try
            # Ajuste GLM (Binomial/Logit)
            res = glm(X_glm, y_bin, Binomial(), LogitLink())

            # Obtener coeficientes absolutos
            all_coefs = abs.(coef(res))
            
            # 3. Quitar el coeficiente del intercepto (es el primero)
            feat_coefs = all_coefs[2:end] 

            # Acumulamos importancia
            importances .+= feat_coefs
            successful_classes += 1

        catch e
            # Ignorar fallos de convergencia puntuales
        end
    end

    if successful_classes == 0
        return zeros(Float64, n_features)
    end

    return importances ./ successful_classes
end


# ========================================================
# FIT: entrenamiento del selector multiclase
# ========================================================

function MMI.fit(model::SelectorRFE, verbosity::Int, X)

    # 1. Verificaciones básicas (CORREGIDO: String)
    if "Activity" ∉ names(X)
        throw(AssertionError("Falta la columna 'Activity' en el dataset."))
    end
    
    # Acceso seguro
    y = X[!, "Activity"]
    
    # 2. Filtrar columnas (CORREGIDO: String)
    feature_cols = filter(c -> c ∉ ["subject", "Activity"], names(X))
    
    # Copia mutable de las features actuales
    current_features = copy(feature_cols)

    # --- CICLO RFE ---
    while length(current_features) > model.k

        # 1. Calcular importancia
        imps = feature_importance_multiclass(X, y, current_features)

        # 2. Cuántas borrar (20% por iteración es más seguro que 50%)
        n_current = length(current_features)
        n_target_removal = max(1, floor(Int, n_current * 0.2))
        
        # Ajuste fino: no borrar más de la cuenta para no bajar de k
        if n_current - n_target_removal < model.k
            n_target_removal = n_current - model.k
        end
        
        # 3. Identificar peores (menor coeficiente)
        # sortperm devuelve índices ordenados de MENOR a MAYOR valor (los peores primero)
        perm_indices = sortperm(imps) 
        worst_indices = perm_indices[1:n_target_removal]

        # 4. Eliminar
        # deleteat! requiere índices ordenados crecientes
        deleteat!(current_features, sort(worst_indices))

        if verbosity > 0
            println("RFE iteración: quedan $(length(current_features)) variables")
        end
    end

    fitresult = (selected=current_features,)
    report = (selected=current_features,)

    return fitresult, nothing, report
end


# ========================================================
# TRANSFORM (CORREGIDO)
# ========================================================

function MMI.transform(model::SelectorRFE, fitresult, X)
    # select con vector de Strings
    return select(X, fitresult.selected)
end

## Proyecciones (Sin projección, PCA, LDA, ICA)

In [19]:
struct NoProjection <: MMI.Unsupervised end
MMI.fit(::NoProjection, ::Int, ::Any) = (nothing, nothing, nothing)
MMI.transform(::NoProjection, ::Nothing, X) = X

In [20]:
struct CustomLDA <: MMI.Unsupervised
    outdim::Int
end
CustomLDA(; outdim=20) = CustomLDA(outdim)
function MMI.fit(model::CustomLDA, verbosity::Int, X)
    # FIX: Filtrar usando Strings
    feature_cols = filter(c -> c ∉ ["subject", "Activity"], names(X))
    
    # Matriz y Target
    X_mat = Matrix{Float64}(X[:, feature_cols])' 
    y_cat = categorical(X.Activity)
    y_int = levelcode.(y_cat)
    
    lda_model = MultivariateStats.fit(MulticlassLDA, X_mat, y_int; outdim=model.outdim)
    return (proj=lda_model, features=feature_cols), nothing, nothing
end
function MMI.transform(model::CustomLDA, fitresult, X)
    X_mat = Matrix{Float64}(X[:, fitresult.features])'
    X_proj = MultivariateStats.predict(fitresult.proj, X_mat)'
    return DataFrame(X_proj, :auto)
end

# --- SelectorPlusLDA (Corregido) ---
struct SelectorPlusLDA <: MMI.Unsupervised
    selector::MMI.Unsupervised
    lda::CustomLDA
end
function MMI.fit(model::SelectorPlusLDA, verbosity::Int, X)
    fit_sel, _, _ = MMI.fit(model.selector, verbosity, X)
    X_sub = MMI.transform(model.selector, fit_sel, X)
    X_sub.Activity = X.Activity # Reinsertar target
    fit_lda, _, _ = MMI.fit(model.lda, verbosity, X_sub)
    return (sel=fit_sel, lda=fit_lda), nothing, nothing
end
function MMI.transform(model::SelectorPlusLDA, fitresult, X)
    X_sub = MMI.transform(model.selector, fitresult.sel, X)
    return MMI.transform(model.lda, fitresult.lda, X_sub)
end

## PIPELINES

| ID | Filtro de características       | Proyección                       | Clasificador | Hiperparámetros            |
| -- | ------------------------------- | -------------------------------- | ------------ | -------------------------- |
| 1  | **Sin filtrado** (SelectorNone) | **Sin reducción** (NoProjection) | MLP          | arquitectura **[50]**      |
| 2  | **ANOVA**                       | **PCA**                          | MLP          | arquitectura **[100]**     |
| 3  | **Mutual Information**          | **ICA**                          | MLP          | arquitectura **[100, 50]** |
| 4  | **Pearson**                     | **LDA**                          | KNN          | **k = 1**                  |
| 5  | **Spearman**                    | **Sin reducción** (NoProjection) | KNN          | **k = 10**                 |
| 6  | **Kendall Tau**                 | **PCA**                          | KNN          | **k = 20**                 |
| 7  | **RFE (Logistic Regression)**   | **PCA**                          | SVM          | **C = 0.1**                |
| 8  | **ANOVA**                       | **LDA**                          | SVM          | **C = 0.5**                |
| 9  | **Sin filtrado** (SelectorNone) | **PCA**                          | SVM          | **C = 1.0**                |


In [21]:
using MLJ
using MLJFlux
using MLJMultivariateStatsInterface
using LIBSVM
using NearestNeighborModels
using DataFrames, CSV, Statistics
import DataFrames: select, Not

# =========================================================
# 2. INSTANCIAS BASE
# =========================================================
NNType  = @load NeuralNetworkClassifier pkg=MLJFlux verbosity=0
KNNType = @load KNNClassifier pkg=NearestNeighborModels verbosity=0
SVCType = @load SVC pkg=LIBSVM verbosity=0
PCAType = @load PCA pkg=MultivariateStats verbosity=0
ICAType = @load ICA pkg=MultivariateStats verbosity=0

# Modelos
mlp_50     = NNType(builder=MLJFlux.MLP(hidden=(50,)), epochs=15)
mlp_100    = NNType(builder=MLJFlux.MLP(hidden=(100,)), epochs=15)
mlp_100_50 = NNType(builder=MLJFlux.MLP(hidden=(100, 50)), epochs=15)

knn_1  = KNNType(K=1)
knn_10 = KNNType(K=10)
knn_20 = KNNType(K=20)

svm_01     = SVCType(cost=0.1, kernel=LIBSVM.Kernel.RadialBasis,tolerance = 1e-5, cachesize=500.0)
svm_05     = SVCType(cost=0.5, kernel=LIBSVM.Kernel.RadialBasis, tolerance = 1e-5, cachesize=500.0)
svm_10     = SVCType(cost=1.0, kernel=LIBSVM.Kernel.RadialBasis, tolerance = 1e-5, cachesize=500.0)

pca_20 = PCAType(maxoutdim=20)
ica_20_relaxed = ICAType(
    outdim = 20,
    maxiter = 300,     # menos iteraciones
    tol = 1.0          # tolerancia MUY relajada
)

custom_lda_20 = CustomLDA(outdim=20)

# =========================================================
# 3. DEFINICIÓN DE PIPELINES P1-P9
# =========================================================

pipelines = Dict()

# RNA
pipelines["P1"] = SelectorNone() |> NoProjection() |> mlp_50
pipelines["P2"] = SelectorANOVA(k=50) |> pca_20 |> mlp_100
pipelines["P3"] = SelectorMI(k=50) |> ica_20_relaxed |> mlp_100_50

# KNN
pipelines["P4"] = SelectorPlusLDA(SelectorPearson(k=50), CustomLDA(outdim=20)) |> knn_1
pipelines["P5"] = SelectorSpearman(k=50) |> NoProjection() |> knn_10
pipelines["P6"] = SelectorKendall(k=50) |> pca_20 |> knn_20

# SVM
pipelines["P7"] = SelectorRFE(k=50) |> pca_20 |> svm_01
pipelines["P8"] = SelectorPlusLDA(SelectorANOVA(k=50), CustomLDA(outdim=20)) |> svm_05
pipelines["P9"] = SelectorNone() |> pca_20 |> svm_10

println(" Pipelines P1-P9 listos y corregidos.")


 Pipelines P1-P9 listos y corregidos.


In [22]:
# =========================================================
# 4. EVALUACIÓN FINAL
# =========================================================

results = DataFrame(ID=[], Description=[], Accuracy=[], F1_Macro=[])

order = ["P1", "P2","P3", "P4", "P5", "P6", "P7", "P8", "P9"]
descriptions = [
    "None + None + MLP50",
    "ANOVA + PCA + MLP100",
    "MI + ICA + MLP100-50",
    "Pearson + LDA + KNN1",
    "Spearman + None + KNN10",
    "Kendall + PCA + KNN20",
    "RFE + PCA + SVM0.1",
    "ANOVA + LDA + SVM0.5",
    "None + PCA + SVM1.0",
]

# Si los SVM son P7-P9:
svm_ids = Set(["P7", "P8", "P9"])


println("Iniciando evaluación unificada...")

for (id, desc) in zip(order, descriptions)
    println("Evaluando $id ($desc)...")
    model = pipelines[id]
    
    accuracies = Float64[]
    f1s = Float64[]

    is_svm = id in svm_ids   # bandera para decidir qué rama usar

    for i in 1:5
        try
            # -----------------------------
            # 1. Cargar datos del fold
            # -----------------------------
            path_tr = "data_processed/folds/fold$(i)_train.csv"
            path_te = "data_processed/folds/fold$(i)_test.csv"
            if !isfile(path_tr)
                path_tr = "../" * path_tr
                path_te = "../" * path_te
            end
            
            df_tr = CSV.read(path_tr, DataFrame)
            df_te = CSV.read(path_te, DataFrame)
            
            # Asegurar que son strings
            df_tr.Activity = String.(df_tr.Activity)
            df_te.Activity = String.(df_te.Activity)

            # -----------------------------
            # 2. Normalización MinMax
            # -----------------------------
            scaler = MyMinMaxScaler(ignore=[:subject, :Activity])
            mach_sc = machine(scaler, df_tr)
            MLJBase.fit!(mach_sc, verbosity=0)
            df_tr_s = MLJBase.transform(mach_sc, df_tr)
            df_te_s = MLJBase.transform(mach_sc, df_te)

            # -----------------------------
            # 3. Target y (rama SVM / no SVM)
            # -----------------------------
            y_tr = nothing
            y_te = nothing

            if is_svm
                # SVM: Activity como OrderedFactor
                df_tr_s = coerce(df_tr_s, :Activity => OrderedFactor)
                df_te_s = coerce(df_te_s, :Activity => OrderedFactor)
                y_tr = df_tr_s.Activity
                y_te = df_te_s.Activity
            else
                # MLP/KNN: Activity como Multiclass normal
                y_tr = coerce(df_tr_s.Activity, Multiclass)
                y_te = coerce(df_te_s.Activity, Multiclass)
            end
            
            # -----------------------------
            # 4. Features X
            # -----------------------------
            X_tr = select(df_tr_s, Not("subject"))
            X_te = select(df_te_s, Not("subject"))
            
            # Coerción explícita de features numéricos
            X_tr = coerce(X_tr, Count => Continuous)
            X_te = coerce(X_te, Count => Continuous)

            # -----------------------------
            # 5. Entrenar y predecir
            # -----------------------------
            mach = machine(model, X_tr, y_tr)
            MLJBase.fit!(mach, verbosity=0)
            
            y_pred = if is_svm
                # SVM → predicción determinista (etiquetas)
                MLJBase.predict(mach, X_te)
            else
                # MLP/KNN → predicción probabilística, nos quedamos con la moda
                predict_mode(mach, X_te)
            end
            
            # -----------------------------
            # 6. Métricas
            # -----------------------------
            push!(accuracies, accuracy(y_pred, y_te))
            push!(f1s, macro_f1score(y_pred, y_te))
            
        catch e
            println(" Error en $id fold $i:")
            showerror(stdout, e)
            println()
            push!(accuracies, 0.0)
            push!(f1s, 0.0)
        end
    end
    
    mean_acc = mean(accuracies)
    mean_f1  = mean(f1s)
    if isnan(mean_acc); mean_acc = 0.0; end
    if isnan(mean_f1);  mean_f1  = 0.0; end

    println("   -> Promedio Acc: $(round(mean_acc, digits=4))")
    push!(results, (id, desc, mean_acc, mean_f1))
end

# -----------------------------
# 7. Tablas separadas
# -----------------------------
println("\n=== 7. Resultados RNA (MLP) ===")
display(filter(r -> r.ID ∈ ["P1","P2","P3"], results))

println("\n=== 8. Resultados KNN ===")
display(filter(r -> r.ID ∈ ["P4","P5","P6"], results))

println("\n=== 9. Resultados SVM ===")
display(filter(r -> r.ID ∈ collect(svm_ids), results))


Iniciando evaluación unificada...
Evaluando P1 (None + None + MLP50)...
   -> Promedio Acc: 0.8172
Evaluando P2 (ANOVA + PCA + MLP100)...
   -> Promedio Acc: 0.8036
Evaluando P3 (MI + ICA + MLP100-50)...
   -> Promedio Acc: 0.8179
Evaluando P4 (Pearson + LDA + KNN1)...
   -> Promedio Acc: 0.764
Evaluando P5 (Spearman + None + KNN10)...
   -> Promedio Acc: 0.7072
Evaluando P6 (Kendall + PCA + KNN20)...
   -> Promedio Acc: 0.689
Evaluando P7 (RFE + PCA + SVM0.1)...
   -> Promedio Acc: 0.7463
Evaluando P8 (ANOVA + LDA + SVM0.5)...
   -> Promedio Acc: 0.8352
Evaluando P9 (None + PCA + SVM1.0)...
   -> Promedio Acc: 0.8785

=== 7. Resultados RNA (MLP) ===


Row,ID,Description,Accuracy,F1_Macro
,Any,Any,Any,Any
1,P1,None + None + MLP50,0.817218,0.787732
2,P2,ANOVA + PCA + MLP100,0.8036,0.782391
3,P3,MI + ICA + MLP100-50,0.817894,0.802121



=== 8. Resultados KNN ===


Row,ID,Description,Accuracy,F1_Macro
,Any,Any,Any,Any
1,P4,Pearson + LDA + KNN1,0.76403,0.765436
2,P5,Spearman + None + KNN10,0.707192,0.691094
3,P6,Kendall + PCA + KNN20,0.689047,0.670989



=== 9. Resultados SVM ===


Row,ID,Description,Accuracy,F1_Macro
,Any,Any,Any,Any
1,P7,RFE + PCA + SVM0.1,0.746264,0.716026
2,P8,ANOVA + LDA + SVM0.5,0.835156,0.827084
3,P9,None + PCA + SVM1.0,0.878479,0.879205


### ENSEMBLE
##### Parte 1

In [23]:
using MLJ
using MLJScikitLearnInterface
using EvoTrees
using MLJModels
using DecisionTree 
using MLJEnsembles

PCA95 = PCAType(variance_ratio=0.95)

PCA(
  maxoutdim = 0, 
  method = :auto, 
  variance_ratio = 0.95, 
  mean = nothing)

In [24]:
AdaBoostStumpClassifier = @load AdaBoostStumpClassifier pkg=DecisionTree verbosity=0

#Modelos de Ensemble

KNN5 = KNNClassifier(K=5)

Bagging10 = EnsembleModel(
    model = KNNClassifier(K=5),
    n = 10,
    bagging_fraction = 0.8
)

Bagging50 = EnsembleModel(
    model = KNNClassifier(K=5),
    n = 50,
    bagging_fraction = 0.8
)

AdaBoost5 = AdaBoostStumpClassifier(n_iter=5)

Evo50 = EvoTreeClassifier(nrounds=50, eta=0.2)
Evo100 = EvoTreeClassifier(nrounds=100, eta=0.2)

#Pipelines de Ensemble
ensemble_pipelines = Dict{String, Any}()
ensemble_pipelines["BAG10"] = SelectorNone() |> PCA95 |> Bagging10
ensemble_pipelines["BAG50"] = SelectorNone() |> PCA95 |> Bagging50
ensemble_pipelines["ADA5"]  = SelectorNone() |> PCA95 |> AdaBoost5
ensemble_pipelines["EVO50"] = SelectorNone() |> PCA95 |> Evo50
ensemble_pipelines["EVO100"]= SelectorNone() |> PCA95 |> Evo100

ProbabilisticPipeline(
  selector_none = SelectorNone(), 
  pca = PCA(
        maxoutdim = 0, 
        method = :auto, 
        variance_ratio = 0.95, 
        mean = nothing), 
  evo_tree_classifier = EvoTreeClassifier(
        loss = :mlogloss, 
        metric = :mlogloss, 
        nrounds = 100, 
        bagging_size = 1, 
        early_stopping_rounds = 9223372036854775807, 
        L2 = 1.0, 
        lambda = 0.0, 
        gamma = 0.0, 
        eta = 0.2, 
        max_depth = 6, 
        min_weight = 1.0, 
        rowsample = 1.0, 
        colsample = 1.0, 
        nbins = 64, 
        tree_type = :binary, 
        seed = 123, 
        device = :cpu), 
  cache = true)

In [25]:
function evaluar_pipeline(model, fold_id)

    # 1. Cargar datasets del fold
    path_tr = "data_processed/folds/fold$(fold_id)_train.csv"
    path_te = "data_processed/folds/fold$(fold_id)_test.csv"

    df_tr = CSV.read(path_tr, DataFrame)
    df_te = CSV.read(path_te, DataFrame)

    df_tr.Activity = String.(df_tr.Activity)
    df_te.Activity = String.(df_te.Activity)

    # 2. Normalización MinMax
    scaler = MyMinMaxScaler(ignore=[:subject, :Activity])
    mach_sc = machine(scaler, df_tr)
    MLJBase.fit!(mach_sc, verbosity=0)

    df_tr_s = MLJBase.transform(mach_sc, df_tr)
    df_te_s = MLJBase.transform(mach_sc, df_te)

    # 3. Target
    y_tr = coerce(df_tr_s.Activity, Multiclass)
    y_te = coerce(df_te_s.Activity, Multiclass)

    # 4. Features
    X_tr = select(df_tr_s, Not([:subject, :Activity]))
    X_te = select(df_te_s, Not([:subject, :Activity]))

    X_tr = coerce(X_tr, Count => Continuous)
    X_te = coerce(X_te, Count => Continuous)

    # 5. Entrenar
    mach = machine(model, X_tr, y_tr)
    MLJBase.fit!(mach, verbosity=0)

    # 6. Predecir (modo porque son modelos probabilísticos)
    y_pred = predict_mode(mach, X_te)

    acc = accuracy(y_pred, y_te)
    f1 = macro_f1score(y_pred, y_te)

    return acc, f1
end

results_ensemble = DataFrame(ID=[], Accuracy=[], F1_Macro=[])

for (id, model) in ensemble_pipelines
    println("Evaluando $id ...")
    accs = Float64[]
    f1s = Float64[]

    for fold in 1:5
        acc, f1 = evaluar_pipeline(model, fold)
        push!(accs, acc)
        push!(f1s, f1)
    end

    push!(results_ensemble, (id, mean(accs), mean(f1s)))
end

display(results_ensemble)


Evaluando BAG10 ...
Evaluando BAG50 ...
Evaluando EVO100 ...
Evaluando EVO50 ...
Evaluando ADA5 ...


Row,ID,Accuracy,F1_Macro
,Any,Any,Any
1,BAG10,0.875953,0.875902
2,BAG50,0.881413,0.881366
3,EVO100,0.832852,0.827901
4,EVO50,0.818115,0.813318
5,ADA5,0.333356,0.165571


#### Parte 2: Entreno modelos anteriores con todo el conjunto de datos

In [26]:
using CSV, DataFrames, MLJ

# ======================================
# Cargar TRAIN COMPLETO (90% individuos)
# ======================================
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)

# Asegurar formato correcto
df_train.Activity = String.(df_train.Activity)

# ======================
# Normalización MinMax
# ======================
scaler_full = MyMinMaxScaler(ignore=[:subject, :Activity])
mach_scaler_full = machine(scaler_full, df_train)
MLJBase.fit!(mach_scaler_full, verbosity=0)

df_train_s = MLJBase.transform(mach_scaler_full, df_train)

# ======================
# Separar X e y
# ======================
X_train_full = DataFrames.select(df_train_s, Not(["subject", "Activity"]))
y_train_full = coerce(df_train_s.Activity, Multiclass)

X_train_full = coerce(X_train_full, Count => Continuous)

# ================================================================================
# LISTA DE MODELOS A ENTRENAR (P1, P4, P9, BAG50, EVO100, ADA5)
# ================================================================================
modelos_finales = Dict(
    "MLP"   => mlp_100_50,
    "KNN"   => knn_1,
    "SVM"   => svm_10,
    "BAG50" => ensemble_pipelines["BAG50"],
    "EVO100"=> ensemble_pipelines["EVO100"],
    "ADA5"  => ensemble_pipelines["ADA5"]
)

# Diccionario donde guardaremos los modelos ya entrenados
modelos_entrenados = Dict{String, Machine}()

# ======================
# ENTRENAMIENTO FINAL
# ======================
for (nombre, modelo) in modelos_finales
    println("Entrenando modelo final: $nombre ...")

    mach = machine(modelo, X_train_full, y_train_full)
    MLJBase.fit!(mach, verbosity=0)

    modelos_entrenados[nombre] = mach
end

println("\n PASO 2.2 COMPLETADO: Modelos entrenados con todo el conjunto TRAIN.")


Entrenando modelo final: SVM ...
Entrenando modelo final: BAG50 ...
Entrenando modelo final: MLP ...
Entrenando modelo final: KNN ...
Entrenando modelo final: EVO100 ...
Entrenando modelo final: ADA5 ...

 PASO 2.2 COMPLETADO: Modelos entrenados con todo el conjunto TRAIN.


#### Parte 3: Modelos compuestos

In [30]:
# Preparar Holdout para luego 
using CSV, DataFrames, MLJ

# Cargar test holdout (10%)
df_test = CSV.read("data_processed/dataset_test.csv", DataFrame)
df_test.Activity = String.(df_test.Activity)

# Aplicar el MISMO scaler que al train completo (¡muy importante!)
df_test_s = MLJBase.transform(mach_scaler_full, df_test)

# Separar X e y
X_test_full  = select(df_test_s,  Not([:subject, :Activity]))
X_test_full = coerce(X_test_full, Count => Continuous)

y_test_full = coerce(df_test_s.Activity, Multiclass)

1094-element CategoricalArray{String,1,UInt32}:
 "LAYING"
 "LAYING"
 "LAYING"
 "LAYING"
 "LAYING"
 "LAYING"
 "LAYING"
 "LAYING"
 "SITTING"
 "SITTING"
 "LAYING"
 "SITTING"
 "SITTING"
 ⋮
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"
 "WALKING"

In [27]:
RandomForestClassifier = @load RandomForestClassifier pkg=DecisionTree verbosity=0

rf_model = RandomForestClassifier(
    n_trees = 500,
    max_depth = 10
)

RandomForestClassifier(
  max_depth = 10, 
  min_samples_leaf = 1, 
  min_samples_split = 2, 
  min_purity_increase = 0.0, 
  n_subfeatures = -1, 
  n_trees = 500, 
  sampling_fraction = 0.7, 
  feature_importance = :impurity, 
  rng = TaskLocalRNG())

In [28]:
using Random
using StatsBase
using MLJ
const MMI = MLJModelInterface

# =========================================================
# Hard Voting Classifier (MLJ Node)
# =========================================================

@mlj_model mutable struct HardVotingClassifier <: MMI.Supervised
    base_model          # modelo base (ej: svm_10)
    n_splits::Int       # número de particiones
    rng::Int            # semilla
end

# ---------------------------------------------------------
# FIT
# ---------------------------------------------------------
function MMI.fit(model::HardVotingClassifier, verbosity::Int, X, y)

    n = nrows(X)
    idx = collect(1:n)

    rng = MersenneTwister(model.rng)
    Random.shuffle!(rng, idx)

    # Particiones equilibradas
    base_size = div(n, model.n_splits)
    extra     = mod(n, model.n_splits)

    split_indices = Vector{Vector{Int}}()
    start = 1
    for k in 1:model.n_splits
        len = base_size + (k <= extra ? 1 : 0)
        push!(split_indices, idx[start:start+len-1])
        start += len
    end

    machines = Machine[]

    for part in split_indices
        X_sub = X[part, :]
        y_sub = y[part]

        base_clone = deepcopy(model.base_model)
        mach = machine(base_clone, X_sub, y_sub)
        MLJBase.fit!(mach, verbosity=0)

        push!(machines, mach)
    end

    fitresult = (machines = machines,)
    cache = nothing
    report = (
        n_splits = model.n_splits,
        rng = model.rng
    )

    return fitresult, cache, report
end

# ---------------------------------------------------------
# PREDICT (votación mayoritaria)
# ---------------------------------------------------------
function MMI.predict(model::HardVotingClassifier, fitresult, Xnew)

    machines = fitresult.machines
    n_models = length(machines)

    preds = [MLJBase.predict(mach, Xnew) for mach in machines]

    n = length(preds[1])
    y_hat = similar(preds[1])

    for i in 1:n
        votes = [preds[j][i] for j in 1:n_models]
        counts = countmap(votes)
        y_hat[i] = argmax(counts)
    end

    return y_hat
end

# =========================================================
# Instanciación del modelo Hard Voting
# =========================================================

hard_voting_model = HardVotingClassifier(
    base_model = svm_10,   # mejor SVM (P9)
    n_splits   = 3,
    rng        = 104
)

HardVotingClassifier(
  base_model = SVC(
        kernel = LIBSVM.Kernel.RadialBasis, 
        gamma = 0.0, 
        cost = 1.0, 
        cachesize = 500.0, 
        degree = 3, 
        coef0 = 0.0, 
        tolerance = 1.0e-5, 
        shrinking = true), 
  n_splits = 3, 
  rng = 104)

In [31]:
using MLJ
using DataFrames

println("=== Construyendo STACKING Ensemble ===")

# =========================================================
# 1. Predicciones de modelos base sobre TRAIN completo
# =========================================================

println("Generando predicciones de modelos base (TRAIN)...")

pred_mlp_train = predict_mode(modelos_entrenados["MLP"], X_train_full)
pred_knn_train = predict_mode(modelos_entrenados["KNN"], X_train_full)
pred_svm_train = MLJBase.predict(modelos_entrenados["SVM"], X_train_full)

# =========================================================
# 2. Meta-dataset (features = predicciones de modelos base)
# =========================================================
mlp_num = levelcode.(pred_mlp_train)
knn_num = levelcode.(pred_knn_train)
svm_num = levelcode.(pred_svm_train)

meta_train = DataFrame(
    mlp = mlp_num,
    knn = knn_num,
    svm = svm_num
)

"""meta_train = DataFrame(
    mlp = pred_mlp_train,
    knn = pred_knn_train,
    svm = pred_svm_train
)"""

y_meta = y_train_full

println("Meta-dataset de stacking creado:")
println(" - Tamaño: ", size(meta_train))
println(" - Columnas: ", names(meta_train))

# =========================================================
# 3. Meta-modelo (MLP)
# =========================================================

meta_mlp = NNType(
    builder = MLJFlux.MLP(hidden=(50,)),
    epochs = 15
)

println("Entrenando meta-modelo (MLP)...")

meta_train = coerce(meta_train, Count => Continuous)
mach_stacking = machine(meta_mlp, meta_train, y_meta)
MLJBase.fit!(mach_stacking, verbosity=0)

println("STACKING Ensemble entrenado correctamente (TRAIN completo).")

# =========================================================
# 4. Evaluación del STACKING en HOLDOUT (10%)
# =========================================================

println("Evaluando STACKING Ensemble en TEST...")

# Predicciones de modelos base sobre TEST
pred_mlp_test = predict_mode(modelos_entrenados["MLP"], X_test_full)
pred_knn_test = predict_mode(modelos_entrenados["KNN"], X_test_full)
pred_svm_test = MLJBase.predict(modelos_entrenados["SVM"], X_test_full)

# Convertir a numérico
mlp_test_num = levelcode.(pred_mlp_test)
knn_test_num = levelcode.(pred_knn_test)
svm_test_num = levelcode.(pred_svm_test)

meta_test = DataFrame(
    mlp = mlp_test_num,
    knn = knn_test_num,
    svm = svm_test_num
)

meta_test = coerce(meta_test, Count => Continuous)

# Predicción final del stacking
y_pred_stack = predict_mode(mach_stacking, meta_test)

# Métricas
acc_stack = accuracy(y_pred_stack, y_test_full)
f1_stack  = macro_f1score(y_pred_stack, y_test_full)

println("STACKING - Accuracy (test): ", round(acc_stack, digits=4))
println("STACKING - F1 Macro (test): ", round(f1_stack, digits=4))


=== Construyendo STACKING Ensemble ===
Generando predicciones de modelos base (TRAIN)...
Meta-dataset de stacking creado:
 - Tamaño: (9205, 3)
 - Columnas: ["mlp", "knn", "svm"]
Entrenando meta-modelo (MLP)...
STACKING Ensemble entrenado correctamente (TRAIN completo).
Evaluando STACKING Ensemble en TEST...
STACKING - Accuracy (test): 0.8684
STACKING - F1 Macro (test): 0.8455


In [32]:
using XGBoost
XGBoostClassifier = @load XGBoostClassifier pkg=XGBoost verbosity=0
xgb_model = XGBoostClassifier()

XGBoostClassifier(
  test = 1, 
  num_round = 100, 
  booster = "gbtree", 
  disable_default_eval_metric = 0, 
  eta = 0.3, 
  num_parallel_tree = 1, 
  gamma = 0.0, 
  max_depth = 6, 
  min_child_weight = 1.0, 
  max_delta_step = 0.0, 
  subsample = 1.0, 
  colsample_bytree = 1.0, 
  colsample_bylevel = 1.0, 
  colsample_bynode = 1.0, 
  lambda = 1.0, 
  alpha = 0.0, 
  tree_method = "auto", 
  sketch_eps = 0.03, 
  scale_pos_weight = 1.0, 
  updater = nothing, 
  refresh_leaf = 1, 
  process_type = "default", 
  grow_policy = "depthwise", 
  max_leaves = 0, 
  max_bin = 256, 
  predictor = "cpu_predictor", 
  sample_type = "uniform", 
  normalize_type = "tree", 
  rate_drop = 0.0, 
  one_drop = 0, 
  skip_drop = 0.0, 
  feature_selector = "cyclic", 
  top_k = 0, 
  tweedie_variance_power = 1.5, 
  objective = "automatic", 
  base_score = 0.5, 
  early_stopping_rounds = 0, 
  watchlist = nothing, 
  nthread = 1, 
  importance_type = "gain", 
  seed = nothing, 
  validate_parameters = 

In [33]:
using MLJ
using LightGBM

LGBMClassifier = @load LGBMClassifier pkg=LightGBM verbosity=0
lgb_model = LGBMClassifier()

[ Info: lib_lightgbm found in system dirs!


LGBMClassifier(
  objective = "multiclass", 
  boosting = "gbdt", 
  num_iterations = 100, 
  learning_rate = 0.1, 
  num_leaves = 31, 
  tree_learner = "serial", 
  num_threads = 0, 
  device_type = "cpu", 
  seed = 0, 
  deterministic = false, 
  force_col_wise = false, 
  force_row_wise = false, 
  histogram_pool_size = -1.0, 
  max_depth = -1, 
  min_data_in_leaf = 20, 
  min_sum_hessian_in_leaf = 0.001, 
  bagging_fraction = 1.0, 
  pos_bagging_fraction = 1.0, 
  neg_bagging_fraction = 1.0, 
  bagging_freq = 0, 
  bagging_seed = 3, 
  feature_fraction = 1.0, 
  feature_fraction_bynode = 1.0, 
  feature_fraction_seed = 2, 
  extra_trees = false, 
  extra_seed = 6, 
  early_stopping_round = 0, 
  first_metric_only = false, 
  max_delta_step = 0.0, 
  lambda_l1 = 0.0, 
  lambda_l2 = 0.0, 
  linear_lambda = 0.0, 
  min_gain_to_split = 0.0, 
  drop_rate = 0.1, 
  max_drop = 50, 
  skip_drop = 0.5, 
  xgboost_dart_mode = false, 
  uniform_drop = false, 
  drop_seed = 4, 
  top_rate = 0.

In [45]:
using PythonCall
using CategoricalArrays
using MLJ
using MLJBase
# =========================================================
# Función auxiliar: evaluación CatBoost en Python
# =========================================================

function eval_catboost_python(X_train_full, y_train_full, X_test_full)
    cb = pyimport("catboost")

    # Pasamos X como matriz numérica -> numpy real
    Xtr_np = Py(Matrix(X_train_full)).to_numpy()
    Xte_np = Py(Matrix(X_test_full)).to_numpy()

    # y como lista de strings (evitas conversiones raras)
    ytr_py = pylist(string.(y_train_full))

    model = cb.CatBoostClassifier()

    model.fit(Xtr_np, ytr_py)

    ypred_py = model.predict(Xte_np)

    # A veces CatBoost devuelve shape (n,1); lo aplanamos a Vector{String}
    ypred_any = pyconvert(Any, ypred_py)

    # CatBoost suele devolver (n, 1)
    if ypred_any isa AbstractMatrix
        ypred_vec = vec([pyconvert(String, ypred_any[i]) for i in eachindex(ypred_any)])
    else
        ypred_vec = [pyconvert(String, y) for y in ypred_any]
    end


    # Convertimos a Categorical con los mismos niveles
    return categorical(ypred_vec, levels=levels(y_train_full))
end


eval_catboost_python (generic function with 1 method)

In [46]:
using DataFrames
using MLJ
using MLJBase
using Tables

# =========================================================
# Tabla de resultados finales
# =========================================================
resultados_test_final = DataFrame(
    Model    = String[],
    Accuracy = Float64[],
    F1_Macro = Float64[]
)

# =========================================================
# 1) MODELOS BASE + ENSEMBLES BÁSICOS (ya entrenados)
# =========================================================
println("\n=== Evaluación modelos base y ensembles básicos ===")

for (nombre, mach) in modelos_entrenados
    println("Evaluando en TEST: $nombre ...")

    y_pred = if nombre == "SVM"
        MLJBase.predict(mach, X_test_full)   # determinista
    else
        predict_mode(mach, X_test_full)      # probabilístico
    end

    acc = accuracy(y_pred, y_test_full)
    f1  = macro_f1score(y_pred, y_test_full)

    push!(resultados_test_final, (nombre, acc, f1))
end

# =========================================================
# 2) RANDOM FOREST
# =========================================================
println("\nEntrenando y evaluando Random Forest...")

mach_rf = machine(rf_model, X_train_full, y_train_full)
MLJBase.fit!(mach_rf, verbosity=0)

y_pred_rf = predict_mode(mach_rf, X_test_full)
acc_rf = accuracy(y_pred_rf, y_test_full)
f1_rf  = macro_f1score(y_pred_rf, y_test_full)

push!(resultados_test_final, ("RandomForest", acc_rf, f1_rf))

# =========================================================
# 3) HARD VOTING
# =========================================================
println("\nEntrenando y evaluando Hard Voting...")

mach_hv = machine(hard_voting_model, X_train_full, y_train_full;scitype_check_level=0)
MLJBase.fit!(mach_hv, verbosity=0)

y_pred_hv = MLJBase.predict(mach_hv, X_test_full)
acc_hv = accuracy(y_pred_hv, y_test_full)
f1_hv  = macro_f1score(y_pred_hv, y_test_full)

push!(resultados_test_final, ("HardVoting", acc_hv, f1_hv))

# =========================================================
# 4) STACKING ENSEMBLE
# =========================================================
println("\nEvaluando Stacking Ensemble...")

push!(resultados_test_final, ("Stacking", acc_stack, f1_stack))

# =========================================================
# 5) BOOSTING AVANZADO
# =========================================================
println("\nEntrenando y evaluando modelos de Boosting avanzado...")

modelos_boost = Dict(
    "XGBoost"   => xgb_model,
    "LightGBM"  => lgb_model,
    "CatBoost"  => cat_model
)

for (nombre, modelo) in modelos_boost
    println("Evaluando $nombre ...")

    if nombre == "CatBoost"
        y_pred = eval_catboost_python(X_train_full, y_train_full, X_test_full)
    else
        mach_boost = machine(modelo, X_train_full, y_train_full; scitype_check_level=0)
        MLJBase.fit!(mach_boost, verbosity=0)
        y_pred = predict_mode(mach_boost, X_test_full)
    end

    acc = accuracy(y_pred, y_test_full)
    f1  = macro_f1score(y_pred, y_test_full)
    push!(resultados_test_final, (nombre, acc, f1))
end

# =========================================================
# RESULTADOS FINALES
# =========================================================
println("\n=== RESULTADOS FINALES EN EL HOLDOUT (10%) ===")
display(resultados_test_final)


=== Evaluación modelos base y ensembles básicos ===
Evaluando en TEST: SVM ...
Evaluando en TEST: BAG50 ...
Evaluando en TEST: MLP ...
Evaluando en TEST: KNN ...
Evaluando en TEST: EVO100 ...
Evaluando en TEST: ADA5 ...

Entrenando y evaluando Random Forest...

Entrenando y evaluando Hard Voting...

Evaluando Stacking Ensemble...

Entrenando y evaluando modelos de Boosting avanzado...
Evaluando CatBoost ...
Learning rate set to 0.088611
0:	learn: 1.4845123	total: 313ms	remaining: 5m 12s
1:	learn: 1.2935014	total: 596ms	remaining: 4m 57s
2:	learn: 1.1523486	total: 927ms	remaining: 5m 8s
3:	learn: 1.0366686	total: 1.3s	remaining: 5m 23s
4:	learn: 0.9392246	total: 1.67s	remaining: 5m 31s
5:	learn: 0.8578100	total: 1.93s	remaining: 5m 19s
6:	learn: 0.7912805	total: 2.19s	remaining: 5m 11s
7:	learn: 0.7342167	total: 2.45s	remaining: 5m 4s
8:	learn: 0.6820336	total: 2.69s	remaining: 4m 55s
9:	learn: 0.6368503	total: 2.92s	remaining: 4m 48s
10:	learn: 0.5962721	total: 3.16s	remaining: 4m 44s

Row,Model,Accuracy,F1_Macro
,String,Float64,Float64
1,SVM,0.917733,0.899966
2,BAG50,0.869287,0.842782
3,MLP,0.886654,0.878289
4,KNN,0.868373,0.845599
5,EVO100,0.83181,0.804743
6,ADA5,0.340037,0.168501
7,RandomForest,0.870201,0.84311
8,HardVoting,0.898537,0.874361
9,Stacking,0.868373,0.845512


#### Visualización de los datos (10% )